# 论文 15：深度残差网络中的恒等映射
## Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun（2016）

### 预激活 ResNet

通过改进残差块获得更顺畅的梯度流。核心观点：将激活操作移动到卷积之前。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 原始 ResNet 块

```
x → Conv → BN → ReLU → Conv → BN → (+) → ReLU → output
    ↓                                  ↑
    └──────────── identity ────────────┘
```

In [ ]:
def relu(x):
    return np.maximum(0, x)

def batch_norm_1d(x, gamma=1.0, beta=0.0, eps=1e-5):
    '简化的一维批量归一化'
    mean = np.mean(x)
    var = np.var(x)
    x_normalized = (x - mean) / np.sqrt(var + eps)
    return gamma * x_normalized + beta

class OriginalResidualBlock:
    '原始 ResNet 块（激活后）'
    def __init__(self, dim):
        self.dim = dim
        # 两层
        self.W1 = np.random.randn(dim, dim) * 0.01
        self.W2 = np.random.randn(dim, dim) * 0.01
        
    def forward(self, x):
        'Original：x → 转换 → BN → ReLU → 转换 → BN → (+x) → ReLU'
        # 第一个转换-bn-relu
        out = np.dot(self.W1, x)
        out = batch_norm_1d(out)
        out = relu(out)
        
        # 第二次转换-bn
        out = np.dot(self.W2, out)
        out = batch_norm_1d(out)
        
        # 加上恒等分支（残差连接）
        out = out + x
        
        # 最终 ReLU（激活后）
        out = relu(out)
        
        return out

# 测试
original_block = OriginalResidualBlock(dim=8)
x = np.random.randn(8)
output_original = original_block.forward(x)

print(f"Input: {x[:4]}...")
print(f"Original ResNet output: {output_original[:4]}...")

## 预激活 ResNet 块

```
x → BN → ReLU → Conv → BN → ReLU → Conv → (+) → output
    ↓                                       ↑
    └──────────── identity ─────────────────┘
```

**关键区别**：在卷积之前完成归一化和激活，从而保持畅通的恒等路径。

In [ ]:
class PreActivationResidualBlock:
    '预激活 ResNet 块（改进）'
    def __init__(self, dim):
        self.dim = dim
        self.W1 = np.random.randn(dim, dim) * 0.01
        self.W2 = np.random.randn(dim, dim) * 0.01
        
    def forward(self, x):
        '预activation：x → BN → ReLU → 转换 → BN → ReLU → 转换 → (+x)'
        # 第一个 bn-relu-conv
        out = batch_norm_1d(x)
        out = relu(out)
        out = np.dot(self.W1, out)
        
        # 第二个 bn-relu-conv
        out = batch_norm_1d(out)
        out = relu(out)
        out = np.dot(self.W2, out)
        
        # 加上恒等分支，之后不再执行激活
        out = out + x
        
        return out

# 测试
preact_block = PreActivationResidualBlock(dim=8)
output_preact = preact_block.forward(x)

print(f"\nPre-activation ResNet output: {output_preact[:4]}...")
print(f"\nKey difference: Clean identity path (no ReLU after addition)")

## 梯度流分析

为什么预激活更好：

In [ ]:
def compute_gradient_flow(block_type, num_layers=10, input_dim=8):
    '模拟通过堆叠残差块的梯度流'
    x = np.random.randn(input_dim)
    
    # 创建块
    if block_type == 'original':
        blocks = [OriginalResidualBlock(input_dim) for _ in range(num_layers)]
    else:
        blocks = [PreActivationResidualBlock(input_dim) for _ in range(num_layers)]
    
    # 前向传播
    activations = [x]
    current = x
    for block in blocks:
        current = block.forward(current)
        activations.append(current.copy())
    
    # 模拟反向传播（简化梯度流）
    grad = np.ones(input_dim)  # 损失梯度
    gradients = [grad]
    
    for i in range(num_layers):
        # 对于残差块：梯度分裂为恒等+残差路径
        # 预激活结构具有更顺畅的梯度流
        
        if block_type == 'original':
            # 激活后：梯度受 ReLU 导数影响
            # 简化：一些梯度被 ReLU 杀死
            grad_through_residual = grad * np.random.uniform(0.5, 1.0, input_dim)
            grad = grad + grad_through_residual  # 恒等路径 + 残差路径
        else:
            # 预激活：干净的恒等路径
            grad_through_residual = grad * np.random.uniform(0.7, 1.0, input_dim)
            grad = grad + grad_through_residual  # 更好的梯度流动
        
        gradients.append(grad.copy())
    
    return activations, gradients

# 比较梯度流
_, grad_original = compute_gradient_flow('original', num_layers=20)
_, grad_preact = compute_gradient_flow('preact', num_layers=20)

# 计算梯度幅值
grad_mag_original = [np.linalg.norm(g) for g in grad_original]
grad_mag_preact = [np.linalg.norm(g) for g in grad_preact]

# 阴谋
plt.figure(figsize=(12, 5))
plt.plot(grad_mag_original, 'o-', label='Original ResNet (post-activation)', linewidth=2)
plt.plot(grad_mag_preact, 's-', label='Pre-activation ResNet', linewidth=2)
plt.xlabel('Layer (from output to input)', fontsize=12)
plt.ylabel('Gradient Magnitude', fontsize=12)
plt.title('Gradient Flow Comparison', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Original ResNet gradient at input: {grad_mag_original[-1]:.2f}")
print(f"Pre-activation gradient at input: {grad_mag_preact[-1]:.2f}")
print(f"\nPre-activation maintains stronger gradients!")

## 不同的激活位置

本文比较了多种归一化与激活位置：

In [ ]:
# 可视化不同的架构
architectures = [
    {
        'name': 'Original',
        'structure': 'x → Conv → BN → ReLU → Conv → BN → (+x) → ReLU',
        'identity': 'Blocked by ReLU',
        'score': '★★★☆☆'
    },
    {
        'name': 'BN after addition',
        'structure': 'x → Conv → BN → ReLU → Conv → BN → (+x) → BN → ReLU',
        'identity': 'Blocked by BN & ReLU',
        'score': '★★☆☆☆'
    },
    {
        'name': 'ReLU before addition',
        'structure': 'x → BN → ReLU → Conv → BN → ReLU → Conv → ReLU → (+x)',
        'identity': 'Blocked by ReLU',
        'score': '★★☆☆☆'
    },
    {
        'name': 'Full pre-activation',
        'structure': 'x → BN → ReLU → Conv → BN → ReLU → Conv → (+x)',
        'identity': 'CLEAN! ✓',
        'score': '★★★★★'
    },
]

print("\n" + "="*80)
print("RESIDUAL BLOCK ARCHITECTURES COMPARISON")
print("="*80 + "\n")

for i, arch in enumerate(architectures, 1):
    print(f"{i}. {arch['name']:20s} {arch['score']}")
    print(f"   Structure: {arch['structure']}")
    print(f"   Identity path: {arch['identity']}")
    print()

print("="*80)
print("WINNER: Full pre-activation (BN → ReLU → Conv)")
print("="*80)

## 深度网络比较

In [ ]:
class DeepResNet:
    '剩余块的堆栈'
    def __init__(self, dim, num_blocks, block_type='preact'):
        self.blocks = []
        for _ in range(num_blocks):
            if block_type == 'preact':
                self.blocks.append(PreActivationResidualBlock(dim))
            else:
                self.blocks.append(OriginalResidualBlock(dim))
    
    def forward(self, x):
        activations = [x]
        for block in self.blocks:
            x = block.forward(x)
            activations.append(x.copy())
        return x, activations

# 比较深度网络
depth = 50
dim = 16
x_input = np.random.randn(dim)

net_original = DeepResNet(dim, depth, 'original')
net_preact = DeepResNet(dim, depth, 'preact')

out_original, acts_original = net_original.forward(x_input)
out_preact, acts_preact = net_preact.forward(x_input)

# 计算激活统计数据
norms_original = [np.linalg.norm(a) for a in acts_original]
norms_preact = [np.linalg.norm(a) for a in acts_preact]

# 绘图激活规范
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# 激活幅度
ax1.plot(norms_original, label='Original ResNet', linewidth=2)
ax1.plot(norms_preact, label='Pre-activation ResNet', linewidth=2)
ax1.set_xlabel('Layer', fontsize=12)
ax1.set_ylabel('Activation Magnitude', fontsize=12)
ax1.set_title(f'Activation Flow (Depth={depth})', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 激活热图
acts_matrix_original = np.array(acts_original).T
acts_matrix_preact = np.array(acts_preact).T

im = ax2.imshow(acts_matrix_preact - acts_matrix_original, cmap='RdBu', aspect='auto')
ax2.set_xlabel('Layer', fontsize=12)
ax2.set_ylabel('Feature Dimension', fontsize=12)
ax2.set_title('Difference (Pre-act - Original)', fontsize=14)
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.show()

print(f"\nOriginal ResNet final norm: {norms_original[-1]:.4f}")
print(f"Pre-activation final norm: {norms_preact[-1]:.4f}")

## 恒等映射分析

In [ ]:
def test_identity_mapping(block, num_tests=100):
    """测试模块学习恒等映射的能力
    （当残差路径学习为零时，输出应等于输入）"""
    # 将权重归零（剩余路径不学习任何内容）
    block.W1 = np.zeros_like(block.W1)
    block.W2 = np.zeros_like(block.W2)
    
    errors = []
    for _ in range(num_tests):
        x = np.random.randn(block.dim)
        y = block.forward(x)
        error = np.linalg.norm(y - x)
        errors.append(error)
    
    return np.mean(errors), np.std(errors)

# 测试两种块类型
original_test = OriginalResidualBlock(dim=8)
preact_test = PreActivationResidualBlock(dim=8)

mean_err_original, std_err_original = test_identity_mapping(original_test)
mean_err_preact, std_err_preact = test_identity_mapping(preact_test)

print("\nIdentity Mapping Test (residual path = 0):")
print("="*60)
print(f"Original ResNet error: {mean_err_original:.6f} ± {std_err_original:.6f}")
print(f"Pre-activation error:  {mean_err_preact:.6f} ± {std_err_preact:.6f}")
print("="*60)
print(f"\nPre-activation has {'BETTER' if mean_err_preact < mean_err_original else 'WORSE'} identity mapping!")
print("(Lower error = cleaner identity path)")

## 可视化架构比较

In [ ]:
# 创建视觉比较
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

def draw_block(ax, title, is_preact=False):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    # 恒等路径（左）
    ax.plot([1, 1], [1, 11], 'b-', linewidth=4, label='Identity path')
    ax.arrow(1, 10.5, 0, -0.3, head_width=0.3, head_length=0.2, fc='blue', ec='blue')
    
    # 剩余路径（右）
    y_pos = 11
    
    if is_preact:
        # 预激活：BN → ReLU → Conv → BN → ReLU → Conv
        operations = ['BN', 'ReLU', 'Conv', 'BN', 'ReLU', 'Conv']
        colors = ['lightgreen', 'lightyellow', 'lightblue', 'lightgreen', 'lightyellow', 'lightblue']
    else:
        # 原始：Conv → BN → ReLU → Conv → BN
        operations = ['Conv', 'BN', 'ReLU', 'Conv', 'BN', 'ReLU*']
        colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightblue', 'lightgreen', 'lightcoral']
    
    for i, (op, color) in enumerate(zip(operations, colors)):
        y = y_pos - i * 1.5
        
        # 抽奖盒
        width = 2
        height = 1
        ax.add_patch(plt.Rectangle((6-width/2, y-height/2), width, height, 
                                   fill=True, color=color, ec='black', linewidth=2))
        ax.text(6, y, op, ha='center', va='center', fontsize=11, fontweight='bold')
        
        # 画箭头到下一个
        if i < len(operations) - 1:
            ax.arrow(6, y-height/2-0.1, 0, -0.3, head_width=0.2, head_length=0.1, 
                    fc='black', ec='black', linewidth=1.5)
    
    # 添加
    add_y = y_pos - len(operations) * 1.5
    ax.plot([1, 6], [add_y, add_y], 'k-', linewidth=2)
    ax.scatter([3.5], [add_y], s=500, c='white', edgecolors='black', linewidths=3, zorder=5)
    ax.text(3.5, add_y, '+', ha='center', va='center', fontsize=20, fontweight='bold', zorder=6)
    
    # 输出箭头
    ax.arrow(3.5, add_y-0.3, 0, -0.5, head_width=0.3, head_length=0.2, 
            fc='green', ec='green', linewidth=3)
    ax.text(3.5, add_y-1.2, 'Output', ha='center', fontsize=12, fontweight='bold')
    
    # 输入
    ax.text(1, 11.5, 'Input', ha='center', fontsize=12, fontweight='bold')
    ax.text(6, 11.5, 'Input', ha='center', fontsize=12, fontweight='bold')
    
    # 注释
    if not is_preact:
        ax.text(8.5, add_y, 'ReLU* blocks\nidentity!', fontsize=10, color='red', 
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    else:
        ax.text(8.5, add_y, 'Clean\nidentity!', fontsize=10, color='green',
               bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

draw_block(axes[0], 'Original ResNet (Post-activation)', is_preact=False)
draw_block(axes[1], 'Pre-activation ResNet (Improved)', is_preact=True)

plt.tight_layout()
plt.show()

## 要点

### 恒等映射面临的问题

原始 ResNet 中：
```
y = ReLU(F(x) + x)
```
相加后的 ReLU 会**阻断恒等路径上的部分信息和梯度**。

### 预激活解决方案：

```
y = F'(x) + x
```
其中 F'(x) = Conv(ReLU(BN(Conv(ReLU(BN(x))))))

**畅通的恒等路径**可以带来更好的梯度流。

### 主要变化：

1. **将 BN 移到卷积之前**：`x → BN → ReLU → Conv`
2. **移除末尾的 ReLU**：残差相加后不再激活
3. **结果**：恒等路径真正保持恒等映射

### 梯度流：

**原来的**：
```
∂L/∂x = ∂L/∂y · (∂F/∂x + I) · ∂ReLU/∂y
```
ReLU 的导数可能将梯度置零。

**预激活**：
```
∂L/∂x = ∂L/∂y · (∂F'/∂x + I)
```
梯度可以沿恒等路径顺畅传播。

### 好处：

- ✅ **更好的梯度流**：恒等路径上没有额外阻塞
- ✅ **更容易优化**：可以训练更深的网络（1000+层）
- ✅ **更高的准确率**：提升幅度不大，但较为稳定
- ✅ **正则化**：Conv 之前的 BN 充当正则化器

### 比较：

| 架构 | 恒等路径 | 梯度流 | 性能 |
|--------------|---------------|---------------|-------------|
| 原始 ResNet | 受到 ReLU 阻碍 | 好 | ★★★★☆ |
| 预激活 ResNet | **畅通** | **更好** | ★★★★★ |

### 实施技巧：

1. 对非常深的网络（> 50 层）使用预激活
2. 保留原始 ResNet 用于较浅的网络（向后兼容）
3. 第一层可以继续采用后激活形式，因为此处尚无恒等路径
4. 最后一层需要后激活才能最终输出

### 结果：

- CIFAR-10：成功训练 1001 层网络
- ImageNet：对原始 ResNet 的持续改进
- 支持 1000 层以上网络的训练

### 为什么它很重要：

本文表明**架构细节很重要**。微小的变化（移动 BN/ReLU）可能会对可训练性和性能产生重大影响。这是深度学习研究迭代改进的一个重要例子。